# IPL First Innings Score Prediction

**Tabular Regression Project** — Predict IPL first innings totals.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: IPL ball-by-ball data (local `dataset/ipl.csv`)
Target: `total` (first innings score)

## 2 · Project Overview

We predict the **first innings total score** in IPL T20 cricket matches given in-match context: current runs, wickets, overs bowled, and recent scoring rate. This is the basic version focusing on core regression techniques for sports analytics.

## 3 · Learning Objectives

1. Work with sports analytics data.
2. Handle cricket-specific categorical features (teams, venues).
3. Engineer run-rate features.
4. Apply regression to live score prediction.
5. Compare model performance on sports data.

## 4 · Problem Statement

Given the current match state (runs, wickets, overs, recent performance), predict the **total first innings score** at the end of 20 overs.

## 5 · Why This Project Matters

- IPL is the world's richest cricket league.
- Live score prediction enhances fan experience.
- Sports analytics is a growing ML application area.
- Teaches domain-specific feature engineering.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Local: `dataset/ipl.csv` |
| **Target** | total (first innings score) |
| **Features** | Teams, venue, runs, wickets, overs, recent performance |

## 7 · Dataset Source and License Notes

- **Source**: IPL match data, commonly from Kaggle.
- **License**: Educational use.
- **Note**: Historical match data across multiple IPL seasons.

## 8 · Environment Setup

In [ ]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
np.random.seed(SEED)

## 11 · Dataset Loading

In [ ]:
data_file = os.path.join(SAVE_DIR, 'dataset', 'ipl.csv')
assert os.path.exists(data_file), f'Not found: {data_file}'
df = pd.read_csv(data_file)
print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 12 · Data Validation

In [ ]:
print('Missing:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

## 13 · Exploratory Data Analysis

In [ ]:
# Find target
target_cands = [c for c in df.columns if any(k in c.lower() for k in ['total','score','target'])]
TARGET = target_cands[0] if target_cands else df.select_dtypes('number').columns[-1]
print(f'Target: {TARGET}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].hist(bins=40, ax=axes[0], edgecolor='black')
axes[0].set_title(f'{TARGET} Distribution')
# Team avg scores
for tc in ['bat_team','batting_team']:
    if tc in df.columns:
        df.groupby(tc)[TARGET].mean().sort_values().tail(8).plot.barh(ax=axes[1])
        axes[1].set_title('Avg Score by Batting Team')
        break
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder
for col in df.columns:
    if 'date' in col.lower() or 'mid' in col.lower():
        df = df.drop(columns=[col])
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() > 100: df = df.drop(columns=[col])
    else: df[col] = LabelEncoder().fit_transform(df[col].fillna('Unknown'))
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

In [ ]:
# Run rate
for oc in ['overs','over']:
    if oc in df.columns:
        rc = [c for c in df.columns if 'run' in c.lower() and c != TARGET]
        if rc: df['current_rr'] = df[rc[0]] / (df[oc] + 0.1)
        break
X = df.drop(columns=[TARGET])
y = df[TARGET]
X = X.replace([np.inf, -np.inf], np.nan)
for col in X.columns:
    X[col] = X[col].fillna(X[col].median() if pd.notna(X[col].median()) else 0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R2={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R2: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Boosting Models

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R2={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R2={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Eval of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R2={m['R2']:.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation

- Current runs and overs bowled dominate predictions.
- Wickets constrain final score potential.
- Team identity provides modest additional signal.
- Run rate in last 5 overs captures innings momentum.

## 26 · Limitations

- No player-level features.
- No pitch/weather data.
- Historical patterns may not persist with roster changes.
- Predictions improve with more overs completed.

## 27 · How to Improve

1. Add player strike rates/bowling economy.
2. Include pitch and weather conditions.
3. Phase-specific models (powerplay, middle, death overs).
4. Use sequential models for over-by-over prediction.

## 28 · Production Considerations

- Needs real-time data feed.
- Low-latency inference for live prediction.
- Gambling regulation compliance.
- Retrain each season.

## 29 · Common Mistakes

1. Including final score components as features (leakage).
2. Not engineering run rate.
3. Using player names directly (too many categories).
4. Ignoring innings phase effects.

## 30 · Mini Challenge

1. Predict only from the 10-over mark.
2. Add powerplay indicator.
3. Compare weekday vs weekend match scores.
4. Build venue-specific models.

## 31 · Final Summary

- Cricket score prediction is an engaging sports ML task.
- In-match state features dominate predictions.
- Gradient-boosting models handle the mixed features well.
- Domain knowledge improves feature engineering significantly.

In [ ]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')